### **1. Install tacoreader**

## **2. Load a TACO compliant dataset**

In [1]:
import autoroot
import tacoreader
from shapely import wkb
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import pathlib

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch import LightningDataModule
from loguru import logger

from src.finetuning.dataloader import Cloud3DDataModule
from src.finetuning.transforms import GeoSatTransform
from src.finetuning.model import UNetAutoencoder


In [2]:
tacoreader.use("pandas")

FOLDER = "/home/emiliano/Documents/3DClouds_data/miniset/pretraining/"
files = list(pathlib.Path(FOLDER+"/goes").glob("*"))                   
goes = tacoreader.load(files)
files = list(pathlib.Path(FOLDER+"/himawari").glob("*"))
himawari = tacoreader.load(files)
files = list(pathlib.Path(FOLDER+"/msg").glob("*"))
msg = tacoreader.load(files)

# Concat
full_dataset = tacoreader.concat([goes, himawari, msg])
dataset = full_dataset.data#.to_pandas()



Loading datasets: 100%|██████████| 8/8 [00:00<00:00, 28.28ds/s]


In [3]:
SPLITS_DICT = {
    "train": {
        "years": np.arange(2004, 2025).tolist(),
        "months": np.arange(1, 13).tolist(),
        "days": np.arange(2, 23).tolist(),
    },
    "val": {
        "years": np.arange(2004, 2025).tolist(),
        "months": np.arange(1, 13).tolist(),
        "days": np.arange(24, 27).tolist(),
    },
    "test": {
        "years": np.arange(2004, 2025).tolist(),
        "months": np.arange(1, 13).tolist(),
        "days": np.arange(28, 32).tolist(),
    },
}

def add_split_column(df: pd.DataFrame, date_col: str = "date", split_col: str = "split") -> pd.DataFrame:
    #df = df.copy()

    # ensure datetime
    df[date_col] = pd.to_datetime(df[date_col])

    y = df[date_col].dt.year
    m = df[date_col].dt.month
    d = df[date_col].dt.day

    # start with NaN / unknown
    split = np.full((len(df),), np.nan, dtype=object)
    #split = pd.Series(pd.NA, index=df.index, dtype="string")

    for name, spec in SPLITS_DICT.items():
        mask = (
            y.isin(spec["years"]) &
            m.isin(spec["months"]) &
            d.isin(spec["days"])
        )
        #split.loc[mask] = name
        split[mask] = name

    df[split_col] = split
    return df


In [4]:
dataset = add_split_column(dataset, date_col="stac:time_start", split_col="split")
dataset

   cloud3d:satellite geoenrich:admin_countries  geoenrich:elevation  \
0               GOES           Ocean/Sea/Lakes                  0.0   
1               GOES           Ocean/Sea/Lakes                  0.0   
2               GOES           Ocean/Sea/Lakes                  0.0   
3               GOES           Ocean/Sea/Lakes                  0.0   
4               GOES           Ocean/Sea/Lakes                  0.0   
..               ...                       ...                  ...   
95              GOES           Ocean/Sea/Lakes                  0.0   
96              GOES           Ocean/Sea/Lakes                  0.0   
97              GOES           Ocean/Sea/Lakes                  0.0   
98              GOES           Ocean/Sea/Lakes                  0.0   
99              GOES           Ocean/Sea/Lakes                  0.0   

    geoenrich:precipitation  geoenrich:temperature  \
0                  1.425387             299.975220   
1                  1.425387            

In [5]:
transform = GeoSatTransform(patch_size=[256, 256])
dm = Cloud3DDataModule(dataset, transform)

2026-03-15 23:00:51.085 | INFO     | src.finetuning.dataloader:__init__:28 - There are 18087 files in taco dataset
2026-03-15 23:00:51.142 | INFO     | src.finetuning.dataloader:__init__:57 - MSG DataModule initialized ...
2026-03-15 23:00:51.143 | INFO     | src.finetuning.dataloader:__init__:58 - Length of train dataset: 12987
2026-03-15 23:00:51.143 | INFO     | src.finetuning.dataloader:__init__:59 - Length of test dataset: 2005
2026-03-15 23:00:51.143 | INFO     | src.finetuning.dataloader:__init__:60 - Length of val dataset: 1375


In [9]:
loader = dm.train_dataloader()
batch = next(iter(loader))

AttributeError: Caught AttributeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/emiliano/anaconda3/envs/minclouds/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 308, in _worker_loop
    data = fetcher.fetch(index)
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/emiliano/anaconda3/envs/minclouds/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 51, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/emiliano/anaconda3/envs/minclouds/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 51, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/home/emiliano/Documents/ISP/Athens_winter_school/ch10-3d-hurricane-generation/src/finetuning/dataloader.py", line 130, in __getitem__
    img_cs = self.df.read(self.idxs[idx]).to_panda

In [6]:
model = UNetAutoencoder(lr=1e-3,)

In [7]:
dir_save = "/data/users/emiliano/3DcloudsData/output/notebooks/"
file_nm = "ae-{epoch:03d}-{val_loss:.4f}"
dir_save+file_nm

'/data/users/emiliano/3DcloudsData/output/notebooks/ae-{epoch:03d}-{val_loss:.4f}'

In [8]:
checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",      # or "train/loss"
    mode="min",
    save_top_k=3,
    save_last=True,         # saves last.ckpt
    filename=file_nm,
)

logger = CSVLogger(save_dir=dir_save, name="cloud_ae")

trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=3, 
                     limit_train_batches=0.01,
                     limit_val_batches=0.01,
                    limit_test_batches=0.01,
                        callbacks=[checkpoint_cb],
                        logger=logger)


/home/emiliano/anaconda3/envs/minclouds/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


MisconfigurationException: No supported gpu backend found!

In [ ]:
trainer.fit(model, datamodule=dm)

In [ ]:
version = "0"
df = pd.read_csv(dir_save+"cloud_ae/version_"+version+"/metrics.csv")
print(df.head())

In [ ]:
# remove rows with NaN val/loss
df.dropna(subset=['val_loss'])

In [ ]:
# get validation curves from csv
import matplotlib.pyplot as plt
plt.plot(df.dropna(subset=['val_loss'])["epoch"], df.dropna(subset=['val_loss'])["val_loss"], label="val_loss")


In [ ]:
# train model for another 3 epochs
trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=3+3, 
                     limit_train_batches=0.01,
                     limit_val_batches=0.01,
                    limit_test_batches=0.01,
                        callbacks=[checkpoint_cb],
                        logger=logger)
trainer.fit(model, datamodule=dm, ckpt_path=checkpoint_cb.best_model_path)

In [ ]:
df = pd.read_csv(dir_save+"cloud_ae/version_"+version+"/metrics.csv")
df.dropna(subset=['val_loss'])

In [ ]:
plt.plot(df.dropna(subset=['val_loss'])["epoch"], df.dropna(subset=['val_loss'])["val_loss"], label="val_loss")


In [ ]:
df

In [ ]:
#look at metrics
trainer.test(model, datamodule=dm)